# AegisFlow IDS – Phase 1: data pipeline and honest baselines (Colab)

Runtime: **CPU is enough** (Runtime → Change runtime type → CPU). Uses about 6–9 GB RAM with the default sampling.

Steps:
1. Get the code
2. Download the original CIC-IDS2017 CSVs and the corrected version
3. Run the 5 baselines on both
4. Compare them and look at the confusion matrices

Every number shown below is computed by this notebook. Copy numbers into your report only from these outputs or from `results/*.json`.

In [ ]:
# 1) Code: clone your repo (if it is private, upload the aegisflow/ folder to Drive instead)
!git clone -b claude/bold-tesla-ezqrrr https://github.com/jagan1344/insightflow-ai.git
%cd insightflow-ai/aegisflow
!pip install -q -r requirements.txt

In [ ]:
# Optional: keep data/results on Google Drive so they survive a runtime reset
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/aegisflow && ln -s /content/drive/MyDrive/aegisflow/raw data/raw

In [ ]:
# 2a) Original CIC-IDS2017 (direct download)
!python data/download.py fetch cicids2017_original
!python data/download.py verify cicids2017_original

2b) **Corrected CIC-IDS2017.** Open the DistriNet page listed in `data/DATASETS.md`, copy the link to the CIC-IDS2017 CSV zip, and paste it below. Or upload the zip to Colab or Drive and use `extract`.

In [ ]:
IMPROVED_URL = ""  # paste the zip link here
if IMPROVED_URL:
    !python data/download.py fetch cicids2017_improved --url "$IMPROVED_URL"
# else: !python data/download.py extract cicids2017_improved /content/drive/MyDrive/<file>.zip
!python data/download.py verify cicids2017_improved

In [ ]:
# 3) Run the benchmark (about 10-25 min per dataset on Colab CPU; the RF and MLP are the slowest)
import logging, sys
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", stream=sys.stdout, force=True)
from aegisflow.config import Phase1Config
from aegisflow.benchmark import run_from_folder, summary_table, per_class_f1_table

cfg = Phase1Config()          # see aegisflow/config.py for every setting
print(cfg)
res_orig = run_from_folder(cfg.data_dir / "cicids2017_original", "cicids2017_original", cfg)
summary_table(res_orig).round(4)

In [ ]:
import json
print(json.dumps(res_orig["cleaning"], indent=1, default=str)[:3000])
print(json.dumps(res_orig["sampling"], indent=1))
per_class_f1_table(res_orig).round(4)

In [ ]:
res_impr = run_from_folder(cfg.data_dir / "cicids2017_improved", "cicids2017_improved", cfg)
summary_table(res_impr).round(4)

In [ ]:
per_class_f1_table(res_impr).round(4)

In [ ]:
# 4) Original vs corrected
from aegisflow.benchmark import compare_versions
compare_versions(res_orig, res_impr).round(4)

In [ ]:
# Confusion matrix (row-normalised) for one model
import numpy as np, plotly.express as px
def plot_cm(res, model="xgboost"):
    m = res["models"][model]; cm = np.array(m["confusion_matrix"], dtype=float)
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig = px.imshow(cmn, x=m["class_names"], y=m["class_names"], text_auto=".2f",
                    labels=dict(x="Predicted", y="True", color="Row %"),
                    title=f"{res['dataset']} – {model} (row-normalised)", color_continuous_scale="Blues")
    fig.show()
plot_cm(res_orig); plot_cm(res_impr)

In [ ]:
# Markdown report for your documentation
!python scripts/compare_phase1.py --original results/phase1_cicids2017_original.json --improved results/phase1_cicids2017_improved.json

In [ ]:
# Optional ablations: each run uses its own results name, so nothing is overwritten
# run_from_folder(cfg.data_dir/"cicids2017_original", "cicids2017_original_smote", Phase1Config(imbalance="smote"))
# run_from_folder(cfg.data_dir/"cicids2017_original", "cicids2017_original_dstport", Phase1Config(keep_dst_port=True))
